# 📓 Semana 2 · Dia 5 — Databricks SQL: dashboards, alertas e queries agendadas

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DAA, DEA (BI) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Relatório SQL publicado com alerta |

---


## 📖 Teoria — Databricks SQL — a camada de BI

Tudo o que você construiu em SQL pode virar **visualização → dashboard → alerta → query agendada**, direto sobre o Unity Catalog, sem cópia para outra ferramenta.

Componentes:
- **SQL Warehouse**: compute de BI (1 na Free, 2X-Small)
- **Queries**: editor SQL com histórico
- **Dashboards**: painéis com visualizações interativas
- **Alerts**: gatilhos em condições (ex.: receita caiu 20%)
- **Schedules**: queries rodando em horários (ex.: 8h diário)


### 💻 Na prática — Criando as queries de negócio

Construa as visualizações de BI sobre o Ouro que será criado na Semana 4 — por ora usamos o Bronze para o dashboard de treino.


In [ ]:
%sql
-- KPI 1: receita por mês
SELECT DATE_TRUNC("month", InvoiceDate) AS mes,
       ROUND(SUM(Quantity * UnitPrice), 2) AS receita
FROM workspace.bronze.vendas_bronze
GROUP BY mes ORDER BY mes

In [ ]:
%sql
-- KPI 2: top produtos
SELECT StockCode, ROUND(SUM(Quantity * UnitPrice), 2) AS receita
FROM workspace.bronze.vendas_bronze
GROUP BY StockCode ORDER BY receita DESC LIMIT 10

### 💻 Na prática — Montando o dashboard

1. Em **Queries**, salve as 2 queries acima.
2. Abra **Dashboards → Create Dashboard**.
3. Adicione as queries; ajuste os tipos de visualização (line para série temporal, bar para ranking).
4. **Publish** para ter um link compartilhável.


### 💻 Na prática — Alertas

Crie um alerta que dispara quando a receita do mês cair.


In [ ]:
%sql
-- Query do alerta: receita do mês corrente vs mês anterior
WITH rec AS (
  SELECT DATE_TRUNC("month", InvoiceDate) mes, SUM(Quantity*UnitPrice) receita
  FROM workspace.bronze.vendas_bronze GROUP BY mes)
SELECT (receita - LAG(receita) OVER (ORDER BY mes)) / LAG(receita) OVER (ORDER BY mes) * 100
       AS variacao_pct
FROM rec ORDER BY mes DESC LIMIT 1

### 💻 Na prática — Queries agendadas

1. Na query salva, clique em **Schedule**.
2. Frequência: diária às 08:00.
3. **Send to**: e-mail (opcional; se o e-mail estiver configurado).
4. Salve. A query rodará no warehouse quando ele estiver ativo.


> 🎯 **Dica de prova**: A prova DAA cobra: tipos de visualização adequados ao dado (linha para tempo, barra para ranking), alertas, e a diferença entre query agendada e dashboard. No DEA, saber que BI consulta o catálogo via SQL Warehouse.


## 🎯 Exercícios de fixação

**1.** Qual visualização usar para série temporal? E para ranking?

**2.** Crie um alerta que dispare quando vendas do dia < 1.000.

**3.** Por que dashboards devem consultar o Ouro (e não o Bronze)?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Visualizações

Série temporal = line chart; ranking = bar chart; distribuição = histogram; composição = stacked bar/pie.

**2.** Alerta de vendas

Query: `SELECT COUNT(*) FROM workspace.bronze.vendas_bronze WHERE InvoiceDate >= current_date()` — Alerta condição `< 1000`.

**3.** Ouro vs Bronze

Ouro é limpo, modelado e estável; Bronze é cru e append-only (pode conter duplicatas/erros). BI sobre Bronze gera números errados.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*